# 02 · Signal node — turn the records into facts the policy can test

**What this notebook does:** adds the second node of RM Copilot. It reads the four records that
`research` fetched and writes one new slot, `signals`: every fact a policy clause's *Applies when*
line tests — counts, flags, `worst_days_late`, months since the last accounts — each one carrying the
record it came from.

**What it deliberately does not do:** decide anything. Signal says *"the latest accounts were 264 days
late (AA filed 2025-12-17)"*, never *"decline"*. Outcomes are the policy node's job (notebook 03).

**Where the model comes in — once.** Almost every signal is counting and date arithmetic, where a model
can only add error. The one exception is a charge's `particulars`: free text such as *"All that freehold
interest in the land and property known as…"*. Turning that into a collateral type is reading, so one
model call does it, and code checks the answer. If no live charge has particulars, the call is skipped
and the node costs zero tokens.

Prerequisites: `.env` has `CH_API` and `ANTHROPIC_API_KEY`; kernel is *Python 3.13 (Lloyds .venv)*.

## 1 · Setup

Same as notebook 01, plus the Anthropic SDK (for the one model call) and Pydantic (to fix the shape of its answer).

In [1]:
import json, re, sys
from datetime import date, timedelta
from pathlib import Path
from typing import Literal, TypedDict

import anthropic
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
GENAI = ROOT / "genai"                   # where mcp_ch/ lives
load_dotenv(ROOT / ".env")
print("repo root:", ROOT)

repo root: /Users/natchalin_/Projects/final_project/Lloyds


## 2 · The state — two new slots

| slot | written by | why |
|---|---|---|
| `as_of` | the caller (input) | the policies measure time from the *assessment date*: "within 6 months", "in the last 3 years". Fixing it makes a run reproducible — same records + same `as_of` → same signals. Left empty, the node uses today. |
| `signals` | `signal` | the facts below |

One change from the plan in `langgraph-shared-state-framing.md`: signal reads `profile` as well as
`filings`, `charges` and `officers`. CON-05 and CON-06 need the incorporation date and the last
accounts' made-up date, and those only live in the profile.

In [2]:
class CopilotState(TypedDict, total=False):
    company_number: str        # input
    as_of:    str              # input, optional — ISO date the assessment is made on
    profile:  dict             # ┐
    filings:  dict             # │ research fills these four
    charges:  dict             # │
    officers: dict             # ┘
    signals:  dict             # signal fills this one

## 3 · The research node — copied from notebook 01, unchanged

This notebook imports nothing from 01; the state is the only contract between them. So the MCP
client and the research node are declared again here, exactly as before.

In [3]:
client = MultiServerMCPClient({
    "companies_house": {
        "transport": "stdio",
        "command":   sys.executable,
        "args":      ["-m", "mcp_ch.server"],
        "cwd":       str(GENAI),
    }
})
TOOL = {t.name: t for t in await client.get_tools()}


def _unwrap(result) -> dict | None:
    # MCP returns [{'type': 'text', 'text': '{...json...}'}]; give back the dict.
    if isinstance(result, list) and result and result[0].get("type") == "text":
        return json.loads(result[0]["text"])
    return result


async def research(state: CopilotState) -> dict:
    n = state["company_number"]
    return {
        "profile":  _unwrap(await TOOL["get_company_profile"].ainvoke({"company_number": n})),
        "filings":  _unwrap(await TOOL["get_filing_history"].ainvoke({"company_number": n})),
        "charges":  _unwrap(await TOOL["get_charges"].ainvoke({"company_number": n})),
        "officers": _unwrap(await TOOL["get_officers"].ainvoke({"company_number": n})),
    }

print("tools:", list(TOOL))

tools: ['get_company_profile', 'get_filing_history', 'get_charges', 'get_officers']


## 4 · Clause → signal: the design, before the code

Each policy clause ends with an *Applies when* line. Signal's job is to make every one of those a
simple lookup for the policy node. Reading the three policy files clause by clause gives this table,
and the code below follows it row by row.

| clause | *Applies when* (paraphrased) | signal |
|---|---|---|
| SEC-01 | an outstanding charge to a third-party lender | `charges.outstanding_third_party` |
| SEC-02 | no charges, or all fully satisfied | `charges.clean_position` |
| SEC-03 | a fully-satisfied charge | `charges.satisfied` |
| SEC-04 | a part-satisfied charge | `charges.part_satisfied` |
| SEC-05 | a third-party charge created in the last 180 days | `charges.third_party_last_180d` |
| SEC-06 | 2+ charges to the same lender within 30 days | `charges.same_lender_within_30d` |
| SEC-07 | a charge containing a negative pledge | `charges.negative_pledge` |
| SEC-08 | a floating charge that is not fully satisfied | `charges.floating_live` |
| CON-01 | accounts filed > 30 days late in the last 3 years | `filings.worst_days_late_3y` |
| CON-02 | the latest accounts filed > 180 days late | `filings.latest_accounts.days_late` |
| CON-03 | 2+ late accounts filings in the last 3 years | `filings.late_3y` |
| CON-04 | latest accounts are micro-entity / total-exemption | `filings.latest_micro_or_exempt` |
| CON-05 | no accounts on record and incorporated > 21 months ago | `filings.accounts_on_record`, `filings.months_since_incorporation` |
| CON-06 | latest accounts made up > 18 months ago | `filings.months_since_made_up` |
| CON-07 | a liquidation filing | `filings.insolvency_filings`, `company.has_insolvency_history` |
| EVD-01 | incorporation date, accounts, full charge position held | `missing` |
| EVD-04 | collateral stated without particulars | `collateral` (only from particulars, and checked) |

**Not computable yet: CON-08 (paper filing).** `mcp_ch/tools.py` does not return the `paper_filed`
field, so the signal is absent rather than guessed. The clause is informational only (PROCEED).

**Every list holds references, not just counts.** `outstanding_third_party` is `["108125710002", …]`, not `2`.
An empty list means "does not apply"; a non-empty one is both the flag and its evidence. That is EVD-02
(every assertion traceable to a record) built into the data shape, so the brief can cite without searching.

## 5 · Helpers — references and dates

A **reference** is how a later node points at a record. Charges use their `charge_code`, but charges
registered before April 2013 have none (`charge_code` is `None`). For those, the reference falls back to
the creation date and the lender, with a `#2` suffix if two would otherwise collide.

In [4]:
LIVE = {"outstanding", "part-satisfied"}          # not yet fully redeemed


def _d(s: str | None) -> date | None:
    return date.fromisoformat(s) if s else None


def _months(earlier: date, later: date) -> float:
    return round((later - earlier).days / 30.44, 1)


def charge_refs(items: list[dict]) -> list[str]:
    # One stable, human-readable reference per charge, in the same order as `items`.
    refs, seen = [], {}
    for c in items:
        lender = (c["persons_entitled"] or ["?"])[0]
        lender = lender if len(lender) <= 28 else lender[:27].rstrip() + "…"
        base = c["charge_code"] or f"created {c['created_on']} · {lender}"
        seen[base] = seen.get(base, 0) + 1
        refs.append(base if seen[base] == 1 else f"{base} #{seen[base]}")
    return refs


def filing_ref(f: dict) -> str:
    return f"{f['type']} filed {f['date']}"

## 6 · Charge signals (SEC-01 … SEC-08)

One detail matters more than it looks: **`None` is not `False`.** The fixed / floating / negative-pledge
flags are only recorded for charges registered since April 2013. On an older charge they are `None`,
meaning *not recorded*, not *no*. Treating `None` as `False` would silently clear SEC-07 and SEC-08 for
exactly the charges we know least about. So those charges get their own list,
`flags_not_recorded_live`, and the policy node decides what an unknown means.

In [5]:
def _same_lender_within_30d(refs: list[str], items: list[dict]) -> list[dict]:
    # SEC-06: runs of 2+ charges to one lender, each created within 30 days of the previous one.
    lenders = {p.strip().lower() for c in items for p in c["persons_entitled"]}
    groups = []
    for lender in sorted(lenders):
        dated = sorted((_d(c["created_on"]), r) for r, c in zip(refs, items)
                       if c["created_on"] and lender in {p.strip().lower() for p in c["persons_entitled"]})
        run = dated[:1]
        for prev, cur in zip(dated, dated[1:]):
            if (cur[0] - prev[0]).days <= 30:
                run.append(cur)
                continue
            if len(run) >= 2:
                groups.append({"lender": lender, "charges": [r for _, r in run]})
            run = [cur]
        if len(run) >= 2:
            groups.append({"lender": lender, "charges": [r for _, r in run]})
    return groups


def charge_signals(charges: dict, as_of: date) -> dict:
    items = charges["items"]
    refs  = charge_refs(items)
    where = lambda test: [r for r, c in zip(refs, items) if test(c)]
    age   = lambda c: (as_of - _d(c["created_on"])).days if c["created_on"] else None
    return {
        "total": len(items),
        "live":  where(lambda c: c["status"] in LIVE),
        "clean_position":          all(c["status"] == "fully-satisfied" for c in items),   # SEC-02 (True if none)
        "satisfied":               where(lambda c: c["status"] == "fully-satisfied"),       # SEC-03
        "outstanding_third_party": where(lambda c: c["status"] == "outstanding"
                                                   and c["lender_group"] == "third_party"),  # SEC-01
        "part_satisfied":          where(lambda c: c["status"] == "part-satisfied"),         # SEC-04
        "third_party_last_180d":   where(lambda c: c["lender_group"] == "third_party"
                                                   and age(c) is not None and 0 <= age(c) <= 180),  # SEC-05
        "same_lender_within_30d":  _same_lender_within_30d(refs, items),                      # SEC-06
        "negative_pledge":         where(lambda c: c["contains_negative_pledge"] is True),    # SEC-07
        "floating_live":           where(lambda c: c["contains_floating_charge"] is True
                                                   and c["status"] != "fully-satisfied"),     # SEC-08
        "flags_not_recorded_live": where(lambda c: c["status"] in LIVE
                                                   and (c["contains_negative_pledge"] is None
                                                        or c["contains_floating_charge"] is None)),
        "own_group":               where(lambda c: c["lender_group"] == "own"),   # should be [] for a lead
    }

## 7 · Filing signals (CON-01 … CON-07)

`days_late` was already computed by the server (notebook 01, section 7), so this is filtering and
taking maxima. Three details:

- **Description wording.** `tools.py` decodes `accounts-with-accounts-type-micro-entity` into
  `accounts with accounts type micro entity`, so the hyphen is gone. The CON-04 pattern matches both spellings.
- **Made-up date comes from the profile.** The filing items fold `made_up_date` into the description text;
  the profile has it as its own field (`last_accounts_made_up_to`).
- **`window_truncated`.** The filing tool returns at most 60 items. For a company with a long history,
  "the last 3 years" is only as complete as that window, and the flag says when it was cut.

In [6]:
MICRO_OR_EXEMPT = re.compile(r"micro[\s-]?entity|total[\s-]?exemption", re.I)
THREE_YEARS = timedelta(days=round(3 * 365.25))


def filing_signals(filings: dict, profile: dict, as_of: date) -> dict:
    items    = filings["items"]
    accounts = sorted((f for f in items if f["category"] == "accounts"),
                      key=lambda f: f["date"], reverse=True)                     # newest first
    recent   = [f for f in accounts if timedelta(0) <= as_of - _d(f["date"]) <= THREE_YEARS]
    latest   = accounts[0] if accounts else None
    worst    = max((f for f in recent if f["days_late"] is not None),
                   key=lambda f: f["days_late"], default=None)
    inc      = _d(profile.get("date_of_creation"))
    made_up  = _d(profile.get("last_accounts_made_up_to"))
    return {
        "accounts_on_record": len(accounts),                                       # CON-05
        "latest_accounts": latest and {"ref": filing_ref(latest),
                                       "days_late": latest["days_late"],           # CON-02
                                       "description": latest["description"]},
        "latest_micro_or_exempt": bool(latest and MICRO_OR_EXEMPT.search(latest["description"])),  # CON-04
        "worst_days_late_3y": worst and {"ref": filing_ref(worst),
                                         "days_late": worst["days_late"]},         # CON-01
        "late_3y": [filing_ref(f) for f in recent if (f["days_late"] or 0) > 0],   # CON-03
        "months_since_incorporation": inc and _months(inc, as_of),                 # CON-05
        "last_made_up_to": profile.get("last_accounts_made_up_to"),
        "months_since_made_up": made_up and _months(made_up, as_of),               # CON-06
        "insolvency_filings": [filing_ref(f) for f in items
                               if f["category"] in ("liquidation", "insolvency")], # CON-07
        "window_truncated": filings["kept"] > len(items),
    }

## 8 · Officer signals

No clause tests officers yet, but the brief will want a line on the board. Counts and role/date references
only — the officer tool already drops personal data, and nothing here adds it back.

In [7]:
def officer_signals(officers: dict, as_of: date) -> dict:
    new = [o for o in officers["items"]
           if o["appointed_on"] and 0 <= (as_of - _d(o["appointed_on"])).days <= 365]
    return {
        "active": officers["active"],
        "total_ever": officers["total"],
        "appointed_last_12m": [f"{o['role']} appointed {o['appointed_on']}" for o in new],
    }

## 9 · The one model call — collateral type from `particulars`

**Why a model here and nowhere else:** particulars are free text written by solicitors, with endless
phrasings for the same thing — *"freehold interest in the land and property known as…"*, *"the account
and the deposit"*, *"over the shares"*. A regex list would never be complete. Reading them is the model's job.

**How it is kept on a short leash:**

1. **Only the particulars text is sent** — no company name, no lender name. EVD-03 bans knowledge about a
   lender's typical business; the easiest way to obey is to never show the model the lender.
2. **The answer has a fixed shape.** `collateral` must be one of nine labels (`Literal`), and the SDK's
   `messages.parse` validates the reply against the Pydantic class.
3. **The model must quote its evidence,** and code checks the quote really appears in the text
   (`verified`). An unverified label stays in the state but must not be used for a collateral statement (EVD-04).
4. **Only live charges** (outstanding / part-satisfied) are sent — a satisfied charge no longer
   encumbers anything. Identical texts are sent once: 36EL's two charges share one description, so that is one item.
5. **One call per company**, all texts batched together, and no call at all when there is nothing to read.

**Model settings:** `claude-opus-5` at `effort: "low"` (a short classification doesn't need deep
thinking). `fallbacks="default"` means that if a safety classifier declines the request, the API re-runs
it on another model instead of returning an empty refusal. Swap `MODEL` to `"claude-sonnet-5"` (as in
`00_hello_claude`) if you'd rather pay less per call.

In [8]:
MODEL = "claude-opus-5"
llm = anthropic.AsyncAnthropic()                  # reads ANTHROPIC_API_KEY from the environment

Collateral = Literal["property", "cash_deposit", "shares", "receivables", "vehicles_plant_equipment",
                     "intellectual_property", "all_assets", "other", "not_stated"]


class Label(BaseModel):
    id: str
    collateral: Collateral
    quote: str = Field(description="The words in the text that name the asset, copied exactly. Empty if not_stated.")


class Labels(BaseModel):
    labels: list[Label]


SYSTEM = (
    "You classify the free-text particulars of UK Companies House charges by the asset they secure. "
    "Use only the text you are given. Do not use knowledge about lenders, companies, or what such "
    "charges usually cover. If the text does not say what is secured (for example it only says "
    "'see image for full details'), answer not_stated. For quote, copy the words from the text that "
    "name the asset, exactly as written."
)


def _norm(s: str) -> str:
    return " ".join(s.lower().split())


async def classify_collateral(charges: dict) -> dict:
    items, refs = charges["items"], charge_refs(charges["items"])
    texts: dict[str, list[str]] = {}                              # particulars text -> charge refs
    for r, c in zip(refs, items):
        if c["status"] in LIVE and c["particulars"]:
            texts.setdefault(c["particulars"], []).append(r)
    if not texts:
        return {}                                                 # nothing to read: no call, zero tokens

    batch = [{"id": f"t{i}", "text": t} for i, t in enumerate(texts, 1)]
    resp = await llm.beta.messages.parse(
        model=MODEL,
        max_tokens=4096,
        system=SYSTEM,
        messages=[{"role": "user", "content": json.dumps(batch, indent=1)}],
        output_format=Labels,
        output_config={"effort": "low"},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    print(f"collateral: 1 call · {len(batch)} text(s) · {resp.model} · "
          f"{resp.usage.input_tokens} in / {resp.usage.output_tokens} out tokens")

    ok = resp.stop_reason != "refusal" and resp.parsed_output is not None
    got = {lab.id: lab for lab in resp.parsed_output.labels} if ok else {}
    out = {}
    for b, charge_list in zip(batch, texts.values()):
        lab = got.get(b["id"])
        entry = ({"collateral": "unclassified", "quote": None, "verified": False} if lab is None else
                 {"collateral": lab.collateral,
                  "quote": lab.quote,
                  "verified": lab.collateral == "not_stated"
                              or (bool(lab.quote) and _norm(lab.quote) in _norm(b["text"]))})
        for r in charge_list:
            out[r] = entry
    return out

## 10 · The signal node

Reads the four records, calls the functions above, returns `{"signals": …}` — one slot, merged into the
state like research's four.

`missing` lists what EVD-01 needs but the records lack (a record that came back empty, no incorporation
date, a charge without a status or lender group). Signal only *reports* the gaps; deciding that they block
a recommendation, and sending the case back to research, is the supervisor's job in notebook 04.

In [9]:
def missing_inputs(state: CopilotState) -> list[str]:
    gaps = [k for k in ("profile", "filings", "charges", "officers") if state.get(k) is None]
    if state.get("profile") is not None and not state["profile"].get("date_of_creation"):
        gaps.append("profile.date_of_creation")
    if state.get("charges") is not None:
        items = state["charges"]["items"]
        gaps += [f"charge {r}: status or lender_group"
                 for r, c in zip(charge_refs(items), items) if not (c["status"] and c["lender_group"])]
    return gaps


async def signal(state: CopilotState) -> dict:
    as_of = _d(state.get("as_of")) or date.today()
    p, f, c, o = (state.get(k) for k in ("profile", "filings", "charges", "officers"))
    return {"signals": {
        "as_of":      as_of.isoformat(),
        "company":    p and {"status": p["company_status"],
                             "accounts_type": p["accounts_type"],
                             "next_accounts_due": p["next_accounts_due"],
                             "accounts_overdue": p["accounts_overdue"],               # CH's own flag, as of today
                             "has_insolvency_history": p["has_insolvency_history"]},   # CON-07
        "charges":    c and charge_signals(c, as_of),
        "filings":    f and filing_signals(f, p or {}, as_of),
        "officers":   o and officer_signals(o, as_of),
        "collateral": await classify_collateral(c) if c else {},
        "missing":    missing_inputs(state),
    }}

## 11 · Build the graph

`START → research → signal → END`. The only new line compared with 01 is the second node and its edge.

In [10]:
g = StateGraph(CopilotState)
g.add_node("research", research)
g.add_node("signal", signal)
g.add_edge(START, "research")
g.add_edge("research", "signal")
g.add_edge("signal", END)
graph = g.compile()

print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| research |   
+----------+   
      *        
      *        
      *        
  +--------+   
  | signal |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## 12 · Run it — 36EL LTD (`10812571`)

`as_of` is fixed so the output is reproducible. The four records come from the server's disk cache, so the
only paid call is the collateral one.

In [11]:
state = await graph.ainvoke({"company_number": "10812571", "as_of": "2026-09-23"})
s = state["signals"]
print(json.dumps(s, indent=2))

collateral: 1 call · 1 text(s) · claude-opus-5 · 723 in / 61 out tokens
{
  "as_of": "2026-09-23",
  "company": {
    "status": "active",
    "accounts_type": "micro-entity",
    "next_accounts_due": "2026-06-30",
    "accounts_overdue": true,
    "has_insolvency_history": false
  },
  "charges": {
    "total": 2,
    "live": [
      "108125710002",
      "108125710001"
    ],
    "clean_position": false,
    "satisfied": [],
    "outstanding_third_party": [
      "108125710002",
      "108125710001"
    ],
    "part_satisfied": [],
    "third_party_last_180d": [],
    "same_lender_within_30d": [
      {
        "lender": "interbay funding limited",
        "charges": [
          "108125710001",
          "108125710002"
        ]
      }
    ],
    "negative_pledge": [
      "108125710002",
      "108125710001"
    ],
    "floating_live": [
      "108125710002",
      "108125710001"
    ],
    "flags_not_recorded_live": [],
    "own_group": []
  },
  "filings": {
    "accounts_on_recor

## 13 · Read the signals the way the policy node will

For each clause, the one value it looks up. This is not a decision — no outcomes are printed — just a check
that every *Applies when* line in section 4 is now a lookup, with the evidence attached.

In [12]:
def clause_view(s: dict) -> None:
    ch, fi = s["charges"], s["filings"]
    rows = [
        ("SEC-01", "outstanding third-party",      ch["outstanding_third_party"]),
        ("SEC-02", "clean position",               ch["clean_position"]),
        ("SEC-04", "part-satisfied",               ch["part_satisfied"]),
        ("SEC-05", "third-party in last 180 days", ch["third_party_last_180d"]),
        ("SEC-06", "same lender within 30 days",   ch["same_lender_within_30d"]),
        ("SEC-07", "negative pledge",              ch["negative_pledge"]),
        ("SEC-08", "floating, not satisfied",      ch["floating_live"]),
        ("  –   ", "flags not recorded (live)",    ch["flags_not_recorded_live"]),
        ("CON-01", "worst days late, 3 years",     fi["worst_days_late_3y"]),
        ("CON-02", "latest accounts days late",    fi["latest_accounts"] and fi["latest_accounts"]["days_late"]),
        ("CON-03", "late filings, 3 years",        fi["late_3y"]),
        ("CON-04", "latest micro / exempt",        fi["latest_micro_or_exempt"]),
        ("CON-05", "accounts on record",           fi["accounts_on_record"]),
        ("CON-06", "months since made up",         fi["months_since_made_up"]),
        ("CON-07", "insolvency filings",           fi["insolvency_filings"]),
        ("EVD-01", "missing",                      s["missing"]),
    ]
    for cid, what, val in rows:
        print(f"{cid}  {what:30s} {val}")
    print("\ncollateral (live charges):")
    for ref, v in s["collateral"].items():
        print(f"  {ref:50s} {v['collateral']:14s} verified={v['verified']}  «{v['quote']}»")
    if fi["window_truncated"]:
        print("\n(!) filing window truncated — 3-year counts may be incomplete")

clause_view(s)

SEC-01  outstanding third-party        ['108125710002', '108125710001']
SEC-02  clean position                 False
SEC-04  part-satisfied                 []
SEC-05  third-party in last 180 days   []
SEC-06  same lender within 30 days     [{'lender': 'interbay funding limited', 'charges': ['108125710001', '108125710002']}]
SEC-07  negative pledge                ['108125710002', '108125710001']
SEC-08  floating, not satisfied        ['108125710002', '108125710001']
  –     flags not recorded (live)      []
CON-01  worst days late, 3 years       {'ref': 'AA filed 2025-12-17', 'days_late': 264}
CON-02  latest accounts days late      264
CON-03  late filings, 3 years          ['AA filed 2025-12-17', 'AA filed 2024-06-27']
CON-04  latest micro / exempt          True
CON-05  accounts on record             7
CON-06  months since made up           26.8
CON-07  insolvency filings             []
EVD-01  missing                        []

collateral (live charges):
  108125710002                

What to notice in 36EL:

- **SEC-06 fires from two charges on the same day to Interbay Funding.** That's a structured facility,
  which only a same-lender + date-window check finds; neither charge looks unusual on its own.
- **The two charges share one particulars text,** so the model read it once and both references got the label.
- **CON-01 and CON-02 point at the same filing here** (AA filed 2025-12-17, 264 days late), because the latest
  accounts were also the latest ones filed. They are still separate signals: CON-01 asks about the worst delay
  in 3 years, CON-02 only about the latest. A company that filed very late in 2024 but on time in 2025 would
  split them.
- **CON-03 counts 2 late filings, not 6.** Four of 36EL's late filings are older than 3 years before `as_of`,
  so they fall outside the window the clause asks about.
- **CON-06: the accounts on record are 26.8 months old,** past the 18-month limit. The profile shows the next
  accounts were due 2026-06-30 and flags them overdue (`company.accounts_overdue`).

## 14 · A second company, for the edge cases — TESCO PLC (`00445790`)

Tesco is not an SME (its accounts type is `group`) and would never be a lead. It's here because it's cached
and it hits three edge cases 36EL doesn't:

- **Pre-2013 charges:** `charge_code` is `None`, so the references fall back to date + lender, and the
  pledge/floating flags are *not recorded*.
- **A long filing history:** 8,000+ filings, cut to the 60 newest, so `window_truncated` is `True`.
- **Different particulars:** deposit accounts rather than property.

In [13]:
tesco = await graph.ainvoke({"company_number": "00445790", "as_of": "2026-09-23"})
clause_view(tesco["signals"])

collateral: 1 call · 2 text(s) · claude-opus-5 · 806 in / 138 out tokens
SEC-01  outstanding third-party        ['created 2009-11-04 · Tesco Trustee Company of Ir…', 'created 2009-11-04 · Tesco Ireland Pension Trust…']
SEC-02  clean position                 False
SEC-04  part-satisfied                 []
SEC-05  third-party in last 180 days   []
SEC-06  same lender within 30 days     []
SEC-07  negative pledge                []
SEC-08  floating, not satisfied        []
  –     flags not recorded (live)      ['created 2009-11-04 · Tesco Trustee Company of Ir…', 'created 2009-11-04 · Tesco Ireland Pension Trust…']
CON-01  worst days late, 3 years       {'ref': 'AA filed 2026-07-25', 'days_late': -126}
CON-02  latest accounts days late      -126
CON-03  late filings, 3 years          []
CON-04  latest micro / exempt          False
CON-05  accounts on record             1
CON-06  months since made up           6.8
CON-07  insolvency filings             []
EVD-01  missing                   

## 15 · Save the state for notebook 03

The state is the only contract between notebooks, so hand it over as a file. Notebook 03 loads these
instead of re-running research and signal — its node gets exactly the input the graph would give it,
with no MCP server and no model call to repeat.

In [14]:
STATE_DIR = ROOT / "genai" / "production" / "state"
STATE_DIR.mkdir(exist_ok=True)
for st in (state, tesco):
    path = STATE_DIR / f"after_signal_{st['company_number']}.json"
    path.write_text(json.dumps(st, indent=1))
    print("saved", path.relative_to(ROOT))

saved genai/production/state/after_signal_10812571.json
saved genai/production/state/after_signal_00445790.json


## What comes next

| notebook | node | reads | adds to state |
|---|---|---|---|
| 03 | `policy` | `signals` | `applicable`, `qualifies`, `evidence_gap` — rules selected in code, applicability judged by the model |
| 04 | `supervisor` + `brief` | `evidence_gap`, `signals`, citations | the retry edge, then `brief` with citations |

Two open items this notebook surfaced:

- **CON-08 needs `paper_filed`** added to `get_filing_history` in `mcp_ch/tools.py`, one line.
- **`window_truncated`** — for companies with long histories, a `max_items` large enough to cover 3 years
  of accounts, or a category filter at the API, would make the CON-01/03 counts complete.